In [1]:
import pandas as pd

In [11]:
df = pd.read_csv("../data/raw/historical_data.csv")
print(f'Datset shape: {df.shape}')
print()
print(f'''Data Types:   
{df.dtypes}''')
print()
print(f'''Missing values: 
{df.isna().sum()}''')
print()
df.head()

Datset shape: (197428, 16)

Data Types:   
market_id                                       float64
created_at                                          str
actual_delivery_time                                str
store_id                                          int64
store_primary_category                              str
order_protocol                                  float64
total_items                                       int64
subtotal                                          int64
num_distinct_items                                int64
min_item_price                                    int64
max_item_price                                    int64
total_onshift_dashers                           float64
total_busy_dashers                              float64
total_outstanding_orders                        float64
estimated_order_place_duration                    int64
estimated_store_to_consumer_driving_duration    float64
dtype: object

Missing values: 
market_id                    

,market_id,created_at,actual_delivery_time,store_id,store_primary_category,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,total_onshift_dashers,total_busy_dashers,total_outstanding_orders,estimated_order_place_duration,estimated_store_to_consumer_driving_duration
0,1.0,2015-02-06 22:24:17,2015-02-06 23:27:16,1845,american,1.0,4,3441,4,557,1239,33.0,14.0,21.0,446,861.0
1,2.0,2015-02-10 21:49:25,2015-02-10 22:56:29,5477,mexican,2.0,1,1900,1,1400,1400,1.0,2.0,2.0,446,690.0
2,3.0,2015-01-22 20:39:28,2015-01-22 21:09:09,5477,NaN,1.0,1,1900,1,1900,1900,1.0,0.0,0.0,446,690.0
3,3.0,2015-02-03 21:21:45,2015-02-03 22:13:00,5477,NaN,1.0,6,6900,5,600,1800,1.0,1.0,2.0,446,289.0
4,3.0,2015-02-15 02:40:36,2015-02-15 03:20:26,5477,NaN,1.0,3,3900,3,1100,1600,6.0,6.0,9.0,446,650.0


There are 197,428 deliveries and 16 columns. Two columns are text, not times. created_at and actual_delivery_time came in as strings, so they'll be converted. Then the target will be total delivery minutes, the gap between them. Money is in cents. A subtotal of 3441 means $34.41. Some store categories are missing, but it's fixable. For example, Store 5477 shows up as "mexican" in row 1 and NaN in rows 2–4, it makes sense to fill the NaN values with the most common category. Row 1 has 2 busy Dashers but only 1 on shift. That seems to be an error, so it might be noise. The number of occurences needs to be counted before deciding what to do. Times are in UTC, and according to the data dictionary the timezone is US/Pacific. Orders appear to cluster around 01:00–04:00 UTC, which is dinnertime on the US West Coast.

In [12]:
print(df.isna().sum().to_string())

market_id                                         987
created_at                                          0
actual_delivery_time                                7
store_id                                            0
store_primary_category                           4760
order_protocol                                    995
total_items                                         0
subtotal                                            0
num_distinct_items                                  0
min_item_price                                      0
max_item_price                                      0
total_onshift_dashers                           16262
total_busy_dashers                              16262
total_outstanding_orders                        16262
estimated_order_place_duration                      0
estimated_store_to_consumer_driving_duration      526


In [15]:
import duckdb
con = duckdb.connect("../data/processed/brightbite.duckdb")
con.execute(open("../sql/01_build_deliveries.sql").read())

In [18]:
checks = """
SELECT
    COUNT(*)                                                AS rows,
    MIN(created_at_utc)                                     AS first_order,
    MAX(created_at_utc)                                     AS last_order,
    SUM(CASE WHEN actual_minutes IS NULL THEN 1 ELSE 0 END) AS no_delivery_time,
    SUM(CASE WHEN busy > onshift THEN 1 ELSE 0 END)         AS busy_gt_onshift,
    SUM(CASE WHEN onshift < 0 OR busy < 0 OR outstanding < 0 THEN 1 ELSE 0 END) AS negative_counts,
    SUM(CASE WHEN store_category = 'unknown' THEN 1 ELSE 0 END) AS still_unknown_category,
    QUANTILE_CONT(actual_minutes, [0.001, 0.01, 0.5, 0.99, 0.999]) AS minute_quantiles,
    MAX(actual_minutes)                                     AS max_minutes
FROM deliveries
"""
print(con.execute(checks).df().T)

                                                                        0
rows                                                               197428
first_order                                           2014-10-19 05:24:15
last_order                                            2015-02-18 06:00:44
no_delivery_time                                                      7.0
busy_gt_onshift                                                   40394.0
negative_counts                                                      81.0
still_unknown_category                                              867.0
minute_quantiles        [14.116666666666667, 19.2, 44.333333333333336,...
max_minutes                                                     141947.65


This will be used to set the cleaning rules. For example, it shows where to cut off impossible delivery times. 

In [19]:
print(con.execute("""
    SELECT COUNT(*) - COUNT(market_id) AS missing_market,
           COUNT(*) - COUNT(onshift)   AS missing_onshift,
           COUNT(*) - COUNT(store_primary_category) AS missing_category
    FROM deliveries
""").df())

   missing_market  missing_onshift  missing_category
0             987            16262              4760


Confirmation that the load matches the missing values found via pandas. 